# 2. LoRA / PEFT Benchmark
Compares full fine-tuning with LoRA on an 8-bit GPT-2 base. This is a deliberately small educational experiment; metrics are not production-quality model evaluation.

In [ ]:
# LoRA vs full fine-tuning on a tiny deterministic finance QA dataset
%pip install -q -U transformers peft accelerate bitsandbytes
import time, math, gc, random
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "This notebook is intentionally GPU-only."
device=torch.device("cuda"); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
MODEL="gpt2"
qa=[("What is EPS in finance?","Earnings per share; net income divided by outstanding shares."),("Define P/E ratio.","Price to earnings; market price per share divided by EPS."),("What is EBITDA?","Earnings before interest, taxes, depreciation and amortization."),("Difference between revenue and profit?","Revenue is total income; profit is revenue minus expenses."),("What is free cash flow?","Cash generated after capital expenditures, available to investors."),("Define market capitalization.","Share price multiplied by total outstanding shares."),("What does ROE measure?","Return on equity measures profit generated relative to shareholders' equity."),("What is a balance sheet?","A statement of assets, liabilities, and shareholders' equity.")]
tok=AutoTokenizer.from_pretrained(MODEL); tok.pad_token=tok.eos_token
MAX_LEN=128
def encode(pair):
    q,a=pair; prompt=f"Question: {q}\nAnswer:"; full=prompt+" "+a+tok.eos_token
    enc=tok(full,add_special_tokens=False,max_length=MAX_LEN,truncation=True)
    qlen=len(tok(prompt,add_special_tokens=False).input_ids)
    labels=enc.input_ids.copy(); labels[:min(qlen,len(labels))]=[-100]*min(qlen,len(labels))
    return {"input_ids":enc.input_ids,"attention_mask":[1]*len(enc.input_ids),"labels":labels}
data=[encode(x) for x in qa]; train=data[:6]; val=data[6:]
def collate(rows):
    m=max(len(x["input_ids"]) for x in rows); out={k:[] for k in rows[0]}
    for r in rows:
        pad=m-len(r["input_ids"]); out["input_ids"].append(r["input_ids"]+[tok.pad_token_id]*pad); out["attention_mask"].append(r["attention_mask"]+[0]*pad); out["labels"].append(r["labels"]+[-100]*pad)
    return {k:torch.tensor(v,device=device) for k,v in out.items()}
@torch.no_grad()
def eval_loss(model):
    model.eval(); losses=[]
    for r in val: losses.append(float(model(**collate([r])).loss))
    return math.exp(sum(losses)/len(losses))

def train(model,steps=20,lr=5e-5):
    model.train(); opt=AdamW([p for p in model.parameters() if p.requires_grad],lr=lr); i=0; ts=[]
    while i<steps:
        r=train[i%len(train)]; t=time.perf_counter(); loss=model(**collate([r])).loss; loss.backward(); opt.step(); opt.zero_grad(); ts.append(time.perf_counter()-t); i+=1
    return sum(ts)/len(ts)

# Full FT
ft=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype=torch.float16).to(device); ft.config.use_cache=False
ft_time=train(ft,steps=20,lr=5e-5); ft_ppl=eval_loss(ft); ft_mem=torch.cuda.max_memory_allocated()/2**20
print("Full FT",ft_time,ft_ppl,ft_mem)
del ft; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# LoRA on 8-bit base
cfg=BitsAndBytesConfig(load_in_8bit=True)
base=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=cfg,device_map="auto"); base=prepare_model_for_kbit_training(base)
lcfg=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,bias="none",task_type="CAUSAL_LM",target_modules=["c_attn","c_proj","c_fc"])
lora=get_peft_model(base,lcfg); lora.print_trainable_parameters()
lora_time=train(lora,steps=20,lr=1e-4); lora_ppl=eval_loss(lora); lora_mem=torch.cuda.max_memory_allocated()/2**20
print("LoRA",lora_time,lora_ppl,lora_mem)
